# AWS Developer Knowledge Graph with mem0, Qdrant, Neo4j, and Claude

This cookbook shows how to build a **three-layer memory pipeline** that turns raw GitHub activity into a structured, queryable knowledge graph — and uses Claude to reason over it.

Unlike plain RAG (retrieve → answer), this pipeline:
- **Extracts structured facts** from raw event text via mem0
- **Stores vector embeddings** in Qdrant for semantic search
- **Builds an entity graph** in Neo4j (developer → PR → repository)
- **Reasons across both layers** using Claude for deep insights

**DEMO_MODE=true by default** — runs fully without GitHub credentials, Qdrant, or Neo4j.

## Install dependencies

In [ ]:
%pip install mem0ai anthropic qdrant-client neo4j ollama --quiet

## Configuration

In [ ]:
import os

DEMO_MODE = os.getenv('DEMO_MODE', 'true').lower() == 'true'
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY', '')
MODEL = 'claude-opus-4-7'

print(f'DEMO_MODE: {DEMO_MODE}')
print(f'API key set: {bool(ANTHROPIC_API_KEY)}')

## Mock GitHub events (used in DEMO_MODE)

In [ ]:
MOCK_EVENTS = [
    'alice merged PR #45 which refactored the auth-service module on 2026-04-11',
    'bob opened PR #46 adding rate limiting to the api-gateway on 2026-04-11',
    'alice reviewed PR #46 and requested changes to the token validation logic',
    'carol committed a TLS certificate rotation fix to the auth-service on 2026-04-12',
    'bob merged PR #47 which added Redis caching to the api-gateway on 2026-04-12',
    'alice opened PR #48 to migrate auth-service database from Postgres to Aurora',
    'dave closed issue #123 — memory leak in the notification-service worker',
    'carol reviewed PR #48 and approved the Aurora migration approach',
]

## Key insight: structured sentences beat raw JSON for mem0 ingestion

Structured natural-language sentences give the LLM better signal for both embedding and entity extraction than raw key-value data.

```python
# Good — preserves context, extracts cleanly
'alice merged PR #45 which refactored the auth module on 2026-04-11'

# Bad — LLM produces noisy, low-quality entities
{"author": "alice", "pr": 45, "repo": "auth-service"}
```

## Set up mem0 with Qdrant + Neo4j (or in-memory for demo)

In [ ]:
from mem0 import Memory

if DEMO_MODE:
    # In-memory config — no external services needed
    memory = Memory()
    print('Running with in-memory store (DEMO_MODE)')
else:
    config = {
        'vector_store': {
            'provider': 'qdrant',
            'config': {'host': 'localhost', 'port': 6333, 'collection_name': 'dev-activity'},
        },
        'graph_store': {
            'provider': 'neo4j',
            'config': {
                'url': os.getenv('NEO4J_URI', 'bolt://localhost:7687'),
                'username': os.getenv('NEO4J_USER', 'neo4j'),
                'password': os.getenv('NEO4J_PASSWORD', 'password'),
            },
        },
        'custom_fact_extraction_prompt': (
            'Only extract entities that are: a specific person (exact username), '
            'a specific repository name, or a specific technical component. '
            'Do NOT extract states, ratios, abstract concepts, or generic words.'
        ),
    }
    memory = Memory.from_config(config)
    print('Running with Qdrant + Neo4j')

## Ingest GitHub events into mem0

In [ ]:
print('Ingesting GitHub events into mem0...')
for event in MOCK_EVENTS:
    memory.add(event, user_id='dev-team')
    print(f'  + {event[:70]}...')

print(f'\nIngested {len(MOCK_EVENTS)} events')

## Query: who has the most context on auth-service?

In [ ]:
results = memory.search('auth-service contributors and changes', user_id='dev-team')
print('Relevant memories:')
for r in results:
    print(f'  - {r["memory"]}')

## Generate developer insight with Claude

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Retrieve facts from mem0
facts = memory.search('auth-service contributors and changes', user_id='dev-team')
context = '\n'.join(f'- {r["memory"]}' for r in facts)

response = client.messages.create(
    model=MODEL,
    max_tokens=512,
    messages=[{
        'role': 'user',
        'content': f'Based on these developer activity facts:\n{context}\n\nWho has the most context on auth-service and why?',
    }],
)

print('Claude insight:')
print(response.content[0].text)

## What to explore next

- Add real GitHub events via `PyGithub` — replace `MOCK_EVENTS` with `repo.get_events()`
- Switch to Qdrant + Neo4j for persistent, queryable storage
- Use mem0's graph store to traverse relationships (`alice → auth-service → PR #48`)
- See the full production pipeline: [mem0-pipeline](https://github.com/TanishkaMarrott/mem0-pipeline)